# CHR-101 / CHR-COLAB-005 matched-seed GA analysis

Use a fresh **free Colab CPU** runtime with no accelerator and no Drive mount. This notebook retrieves one reviewed CHR-38 evaluation CSV and the analysis code from the private repository at an explicit full commit. It does not clone the repository, push results, or save credentials.

The current CHR-101 Jira assignee is the sole official runner and publisher. Other executions are exploratory. The notebook retains failed runs, keeps the offering-level lexicographic fitness dimensions separate, verifies that unguided fitness did not use historical guidance, and compares independently post-scored outcome HPA. It does not turn synthetic or unreviewed output into a research claim.

## 1. Pin and retrieve the reviewed input

The input commit must contain `experiments/CHR-101-CHR-COLAB-005/<run-id>/evaluation_batch.csv`, a machine-readable sibling `evaluation_manifest.json`, and a reviewed sibling `README.md`. The manifest binds the CSV to the backend generator commit, environment, run order, timing controls, and scorer/rule-set provenance. Enter a least-privilege GitHub contents-read token only in the hidden prompt. The input commit and later result commit are intentionally separate.

In [ ]:
import hashlib
import json
import os
import platform
import re
import sys
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
from shutil import make_archive, which
from subprocess import run
from urllib.request import HTTPRedirectHandler, Request, build_opener
from uuid import uuid4
from google.colab import files

if 'run_root' in globals():
    raise RuntimeError('Restart the runtime to begin a new analysis run.')
if sys.version_info < (3, 12):
    raise RuntimeError('Python 3.12 or later is required; select a compatible runtime.')
if input('Confirm free Colab CPU runtime with no accelerator (type YES): ').strip() != 'YES':
    raise ValueError('A free Colab CPU runtime is required.')
gpu = run(['nvidia-smi', '-L'], capture_output=True, text=True) if which('nvidia-smi') else None
if (gpu and gpu.returncode == 0 and gpu.stdout.strip()) or os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError('Disable GPU/TPU acceleration and restart.')

repository = 'saiimonn/Chronos'
commit = input('Explicit 40-character input commit SHA: ').strip().lower()
evaluation_path = input('Reviewed evaluation CSV repository path: ').strip()
publisher = input('Current CHR-101 assignee Jira account ID (check Jira first): ').strip()
if not re.fullmatch(r'[0-9a-f]{40}', commit):
    raise ValueError('A full 40-character commit SHA is required; branches and tags are not allowed.')
if not re.fullmatch(r'experiments/CHR-101-CHR-COLAB-005/[A-Za-z0-9][A-Za-z0-9_.-]*/evaluation_batch\.csv', evaluation_path):
    raise ValueError('Use the reviewed CHR-101 evaluation_batch.csv path.')
if not re.fullmatch(r'[A-Za-z0-9:_-]+', publisher):
    raise ValueError('Enter the assignee Jira account ID, not a token or URL.')
if input('Confirm you are the current CHR-101 Jira assignee (type YES): ').strip() != 'YES':
    raise ValueError('Only the experiment ticket assignee can execute an official run.')

run_id = 'CHR-COLAB-005-' + uuid4().hex
run_root = Path('/content') / run_id
backend_dir = run_root / 'backend'
input_dir = run_root / 'input'
output_dir = run_root / 'analysis'
backend_dir.mkdir(parents=True, exist_ok=False)
input_dir.mkdir(parents=True, exist_ok=False)
code_paths = (
    'backend/requirements.txt',
    'backend/scripts/colab_prototype/analyze_matched_ga.py',
)
retrieved_sha256 = {}

class NoRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

token = getpass('GitHub contents-read token: ')
if not token or any(character.isspace() for character in token):
    raise ValueError('Enter one non-empty token without whitespace.')
opener = build_opener(NoRedirect())
def retrieve(repository_path, destination):
    url = f'https://api.github.com/repos/{repository}/contents/{repository_path}?ref={commit}'
    request = Request(url, headers={
        'Authorization': f'Bearer {token}',
        'Accept': 'application/vnd.github.raw+json',
        'X-GitHub-Api-Version': '2022-11-28',
    })
    with opener.open(request, timeout=120) as response:
        content = response.read()
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)
    retrieved_sha256[repository_path] = hashlib.sha256(content).hexdigest()

for repository_path in code_paths:
    retrieve(repository_path, run_root / repository_path)
evaluation_csv = input_dir / 'evaluation_batch.csv'
input_readme_path = evaluation_path.rsplit('/', 1)[0] + '/README.md'
generation_manifest_path = evaluation_path.rsplit('/', 1)[0] + '/evaluation_manifest.json'
generation_manifest = input_dir / 'evaluation_manifest.json'
retrieve(evaluation_path, evaluation_csv)
retrieve(generation_manifest_path, generation_manifest)
retrieve(input_readme_path, input_dir / 'README.md')
del token
started_at_utc = datetime.now(timezone.utc).isoformat()
print(f'Retrieved {len(retrieved_sha256)} pinned files for {run_id}.')

## 2. Install the pinned environment

Install only from the retrieved requirements file. The package inventory is saved with the checkpoint.

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements.txt'], cwd=backend_dir, check=True)
installed_packages = run(
    [sys.executable, '-m', 'pip', 'freeze'],
    cwd=backend_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
print('Pinned dependencies installed.')

## 3. Validate and analyze all matched runs

The command fails if any seed, mode, failed run, input fingerprint, configuration, or generation history is missing or inconsistent. It writes tables, vector figures, and a hash manifest to a new output directory.

In [ ]:
analysis_script = backend_dir / 'scripts/colab_prototype/analyze_matched_ga.py'
analysis_started_at_utc = datetime.now(timezone.utc).isoformat()
run(
    [
        sys.executable,
        str(analysis_script),
        '--input',
        str(evaluation_csv),
        '--generation-manifest',
        str(generation_manifest),
        '--output-dir',
        str(output_dir),
    ],
    cwd=backend_dir,
    check=True,
)
analysis_completed_at_utc = datetime.now(timezone.utc).isoformat()
print('Analysis complete. Review the tables and figures before downloading.')

In [ ]:
import pandas as pd
from IPython.display import SVG, display

display(pd.read_csv(output_dir / 'mode_summary.csv'))
display(pd.read_csv(output_dir / 'descriptive_statistics.csv'))
display(pd.read_csv(output_dir / 'matched_seed_differences.csv'))
for figure_name in ('metric_boxplots.svg', 'matched_seed_hpa.svg', 'fitness_convergence.svg'):
    display(SVG(filename=str(output_dir / figure_name)))

## 4. Download the review checkpoint

The ZIP excludes the source CSV because the pinned input commit already identifies it. Review locally before publishing approved files under the same CHR-101 run directory. Never commit this ZIP or saved notebook outputs.

In [ ]:
(output_dir / 'installed_packages.txt').write_text(installed_packages, encoding='utf-8')
analysis_manifest = output_dir / 'analysis_manifest.json'
run_metadata = {
    'traceability': 'CHR-101 / CHR-COLAB-005',
    'run_id': run_id,
    'input_commit': commit,
    'input_path': evaluation_path,
    'input_sha256': retrieved_sha256[evaluation_path],
    'generation_manifest_path': generation_manifest_path,
    'generation_manifest_sha256': retrieved_sha256[generation_manifest_path],
    'analysis_code_sha256': retrieved_sha256[code_paths[1]],
    'retrieved_files_sha256': dict(sorted(retrieved_sha256.items())),
    'declared_runner': {
        'source': 'jira_assignee',
        'ticket': 'CHR-101',
        'account_id': publisher,
    },
    'started_at_utc': started_at_utc,
    'analysis_started_at_utc': analysis_started_at_utc,
    'analysis_completed_at_utc': analysis_completed_at_utc,
    'runtime': {
        'python': platform.python_version(),
        'platform': platform.platform(),
        'processor': platform.processor(),
    },
    'analysis_manifest_sha256': hashlib.sha256(analysis_manifest.read_bytes()).hexdigest(),
    'publication_note': 'The result PR must verify the current Jira assignee and reviewed input provenance.',
}
(output_dir / 'run_metadata.json').write_text(
    json.dumps(run_metadata, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)
analysis_manifest_data = json.loads(analysis_manifest.read_text(encoding='utf-8'))
for filename, expected_sha256 in analysis_manifest_data['outputs_sha256'].items():
    actual_sha256 = hashlib.sha256((output_dir / filename).read_bytes()).hexdigest()
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f'Analysis output changed before packaging: {filename}')
checkpoint_files = sorted(path for path in output_dir.iterdir() if path.is_file())
checkpoint_manifest = {
    'schema_version': 1,
    'traceability': 'CHR-101 / CHR-COLAB-005',
    'files_sha256': {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in checkpoint_files},
}
(output_dir / 'checkpoint_manifest.json').write_text(
    json.dumps(checkpoint_manifest, indent=2, sort_keys=True) + '\n',
    encoding='utf-8',
)
archive = Path(make_archive(str(run_root / run_id), 'zip', root_dir=output_dir))
files.download(str(archive))
print(f'Downloaded {archive.name}. Review it locally; the notebook does not publish results.')